# GPT prompt parameter experiments


In [ ]:
%pip install -q openai==0.28.0 pandas


In [ ]:
import os, openai, pandas as pd, json
openai.api_key=os.environ["OPENAI_API_KEY"]

MODELS=["gpt-3.5-turbo-0301","gpt-4-0314"]

SYSTEM_CONTEXT=(
    "The Fridge Freezer in Stainless Steel is a spacious and stylish appliance designed for your kitchen. "
    "It has an energy efficiency rating of C. This fridge freezer offers a generous total storage capacity "
    "of 539 litres, with 387 litres allocated for the fridge and 152 litres for the freezer compartment. "
    "The fridge's matte stainless-steel finish has an anti-fingerprint coating. With a noise level of 35 dB(A), "
    "it operates quietly. The appliance dimensions are 1825 x 840 x 745 mm (H x W x D)."
)
QUESTION="What is the energy rating of the Fridge Freezer, and can it be considered an energy-efficient product?"

BASE={
    "top_p":1.0,
    "n":1,
    "max_tokens":512,
    "stream":False,
    "presence_penalty":0.1,
    "frequency_penalty":0.5,
}

def ask(model,temperature=0.5,frequency_penalty=0.5):
    response=openai.ChatCompletion.create(
        model=model,
        messages=[
            {"role":"system","content":"Answer using the supplied product context and do not invent product facts."},
            {"role":"user","content":f"Context:\n{SYSTEM_CONTEXT}\n\nQuestion:\n{QUESTION}"}
        ],
        temperature=temperature,
        top_p=BASE["top_p"],
        n=BASE["n"],
        max_tokens=BASE["max_tokens"],
        stream=BASE["stream"],
        presence_penalty=BASE["presence_penalty"],
        frequency_penalty=frequency_penalty,
    )
    return response.choices[0].message["content"]


In [ ]:
temperature_rows=[]
for model in MODELS:
    for temp in [0,1,2]:
        temperature_rows.append({
            "model":model,
            "temperature":temp,
            "response":ask(model,temperature=temp,frequency_penalty=0.5),
        })
pd.DataFrame(temperature_rows).to_csv("/content/temperature_results.csv",index=False)
display(pd.DataFrame(temperature_rows))


In [ ]:
frequency_rows=[]
for model in MODELS:
    for fp in [0,1,2]:
        frequency_rows.append({
            "model":model,
            "frequency_penalty":fp,
            "response":ask(model,temperature=0.5,frequency_penalty=fp),
        })
pd.DataFrame(frequency_rows).to_csv("/content/frequency_penalty_results.csv",index=False)
display(pd.DataFrame(frequency_rows))
